# 02 · Estimate elasticity and evaluate predictions

The statistical model answers an interpretable question; XGBoost is a forecasting challenger. Neither forecast accuracy nor a negative price coefficient establishes causality.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
from retail_pricing.config import DATA, ARTIFACTS, REPORTS
observations = pd.read_parquet(DATA / "processed/observations.parquet")
products = pd.read_parquet(DATA / "processed/products.parquet")
quality = json.loads((REPORTS / "data_quality.json").read_text())

from retail_pricing.modeling import split_calendar, fit_elasticity, predict_elasticity, load_artifacts
artifacts, summary = load_artifacts()
parts = split_calendar(observations)
display(pd.DataFrame([{ "split": name, "start": frame.week.min(), "end": frame.week.max(), "rows": len(frame)} for name, frame in parts.items()]))

,split,start,end,rows
0,train,2009-01-14,2011-01-05,346980
1,validation,2011-01-12,2011-07-06,91271
2,test,2011-07-13,2012-01-04,86490


## Refit one product to inspect the estimator
This cell fits only training rows. Store intercepts, support flags and calendar effects adjust observed associations. The full pipeline uses the identical function for every product.

In [2]:
product_id = next(p for p, d in summary["diagnostics"].items() if not d["ineligibility_reasons"])
sample = parts["train"].query("product_id == @product_id")
model = fit_elasticity(sample, observations.week.min())
display(pd.Series(model["diagnostics"]).to_frame("value"))
print("Training smearing factor:", model["smearing"])

,value
elasticity,-0.598935
elasticity_ci,"[-0.7695777695043042, -0.42829153726386654]"
r_squared_log,0.327477
training_rows,6089
zero_sales_excluded,0
rank_deficient,False
condition_number,158.104173
distinct_prices,126
price_log_std,0.159633
training_weeks,104


Training smearing factor: 1.1634137314312667


## Compare all models on common rows
MAE is in units. WAPE is scale-independent at the aggregate level. Positive bias means overprediction. The baseline updates from prior observed weeks, never current-week sales.

In [3]:
scores = [{"split": split, "model": name, **score} for split in ["validation", "test"] for name, score in summary["metrics"][split]["overall"].items()]
display(pd.DataFrame(scores))
px.bar(pd.DataFrame(scores), x="model", y="wape", color="split", barmode="group", title="Held-out forecasting error").show(renderer="plotly_mimetype")

,split,model,mae,wape,bias,rows
0,validation,baseline,9.154229,0.466418,-0.018757,90812
1,validation,log_log,6.672386,0.339965,-0.153903,90812
2,validation,xgboost,8.714578,0.444017,0.469135,90812
3,test,baseline,7.923985,0.433732,0.104935,85742
4,test,log_log,7.395635,0.404812,0.119311,85742
5,test,xgboost,8.634604,0.472629,0.446318,85742


## Inspect errors instead of only one headline score
Model ranking may vary by product, and aggregate performance can conceal weak products. The test period is reported after model choices are fixed.

In [4]:
predictions = pd.read_parquet(ARTIFACTS / summary["version"] / "predictions.parquet")
selected = predictions.query("product_id == @product_id and split == 'test'")
weekly = selected.groupby("week", as_index=False)[["units", "baseline", "log_log", "xgboost"]].sum()
px.line(weekly, x="week", y=["units", "baseline", "log_log", "xgboost"], title="Selected product: test-period units across stores").show(renderer="plotly_mimetype")
display(pd.DataFrame(summary["metrics"]["test"]["by_product"][product_id]).T)

,mae,wape,bias,rows
baseline,1.639349,0.448305,-0.065106,1751.0
log_log,1.568763,0.429003,-0.076381,1751.0
xgboost,2.903019,0.793876,2.439455,1751.0


In [5]:
diagnostics = pd.DataFrame(summary["diagnostics"]).T
print("Modeled products:", len(diagnostics))
print("Products passing evidence checks:", diagnostics.ineligibility_reasons.map(len).eq(0).sum())
display(diagnostics[["elasticity", "elasticity_ci", "ineligibility_reasons"]].head(12))

Modeled products: 55
Products passing evidence checks: 34


,elasticity,elasticity_ci,ineligibility_reasons
1111009477,-1.131219,"[-1.3220711051032017, -0.940366521867871]",[Statistical model did not beat the rolling ba...
1111009497,-0.873453,"[-1.0681269594012954, -0.67877930533276]",[Statistical model did not beat the rolling ba...
1111009507,-1.010416,"[-1.3993054694411806, -0.621526653173448]",[Statistical model did not beat the rolling ba...
1111035398,-0.598935,"[-0.7695777695043042, -0.42829153726386654]",[]
1111038078,-1.307096,"[-1.4229818011745108, -1.191209670904012]",[]
1111038080,-1.533429,"[-1.662846107382406, -1.404012500431745]",[]
1111085319,-2.243832,"[-2.49702671059517, -1.990638253089204]",[Statistical model did not beat the rolling ba...
1111085345,-1.85945,"[-2.0341660684665523, -1.6847342312108256]",[Statistical model did not beat the rolling ba...
1111085350,-1.51057,"[-1.6577289029047741, -1.3634102633883827]",[]
1111087395,-2.43045,"[-2.7086199694127013, -2.152279113852393]",[]


## Interpretation
Use measured results even when the simpler model wins. Eligibility is set using training diagnostics and validation performance, never test performance. The fitted model stays frozen for the demonstration; refitting would require a separately versioned evaluation. See `docs/methodology.md`.